# Phase 3 — training the projector and decision head

Launcher for Phase 3 of Laya-Vision (see `docs/PROJECT_SPEC.md`, `docs/phase3_design.md`, `DECISIONS.md`).
SigLIP and the Laya encoder stay frozen; the 2-layer projector and the decision head (28.1M parameters) train over the
Phase 2 shards with upstream's RLCD objective, bf16, modality dropout 40/30/30. Every run is logged in
`experiments.md`. Gate 3: val accuracy well above majority class and near/above Route A (0.501).

Needs a GPU (L4 recommended; ~30 ex/s → 60k × 3 epochs ≈ 2 h). Paths assume the Drive layout in `DECISIONS.md`.

In [ ]:
# Phase 3 — setup: mount Drive, clone the public repo. Needs a GPU (L4 recommended).
import os
os.chdir("/content")

from google.colab import drive
drive.mount("/content/drive")

REPO = "/content/laya-vision"
if not os.path.exists(REPO):
    !git clone -q https://github.com/savita10/laya-vision.git {REPO}
else:
    !git -C {REPO} fetch -q origin && git -C {REPO} reset --hard origin/main
os.chdir(REPO)
!git log --oneline -1

!pip install -q "transformers==4.49.0" "laya==0.3.24" accelerate datasets pillow

D = "/content/drive/MyDrive/laya-vision"
P1, P2, P3, STEP5 = f"{D}/phase1", f"{D}/phase2", f"{D}/phase3", f"{D}/step5"
for p in (P1, P2, P3): os.makedirs(p, exist_ok=True)

import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## Pre-flight: token budget and equivalence
The budget check reports how much OCR text the 512-token context truncates (result: 8% of pages lose ≥ 50%, no flag).
The equivalence test proves that with no image the model matches upstream `DecisionModel.forward` exactly, and that
gradients reach only the projector and head. The `--diag-equivalence` run verifies this on the GPU under every
attention / precision / compile combination (the compile-only divergence found here is recorded in `DECISIONS.md`).

In [ ]:
!python scripts/ocr_token_budget.py --base-ckpt {STEP5}/laya_finetuned --shards-root {P2}/shards --records {P1}/splits/train.jsonl --out {P3}/ocr_budget_train.json
!python tests/test_vision_equivalence.py --base-ckpt {STEP5}/laya_finetuned
!python scripts/train_vision.py --base-ckpt {STEP5}/laya_finetuned --shards-root {P2}/shards --records-dir {P1}/splits --out-dir {P3}/diag --diag-equivalence

## Tiny and medium runs
Tiny (2k × 1 epoch) proves the loop learns; interrupt it after ~40 updates and re-run to confirm resume. Medium
(10k × 2) shows the image channel taking off (image-only 0.07 → 0.74) and the text channel staying flat (~0.26).

In [ ]:
!python scripts/train_vision.py --base-ckpt {STEP5}/laya_finetuned --shards-root {P2}/shards --records-dir {P1}/splits \
    --comparison-set /content/laya-vision/splits/val_comparison_2000_seed11.json --limit 2000 --epochs 1 --save-every 20 --out-dir {P3}/tiny
!python scripts/train_vision.py --base-ckpt {STEP5}/laya_finetuned --shards-root {P2}/shards --records-dir {P1}/splits \
    --comparison-set /content/laya-vision/splits/val_comparison_2000_seed11.json --limit 10000 --epochs 2 --out-dir {P3}/medium

## Full baseline run and the untrained text baseline
run01_base: 60k × 4 epochs, 32 tokens, projector lr 1e-3 → monitor combined 0.849 / image 0.842 / text 0.306.
The untrained text-only baseline (no `--run-dir`, `--views text`) scores 0.262 — the reference that shows the text
channel barely learns.

In [ ]:
!python scripts/train_vision.py --base-ckpt {STEP5}/laya_finetuned --shards-root {P2}/shards --records-dir {P1}/splits \
    --comparison-set /content/laya-vision/splits/val_comparison_2000_seed11.json --epochs 4 --out-dir {P3}/run01_base
!python scripts/eval_vision.py --base-ckpt {STEP5}/laya_finetuned --shards-root {P2}/shards --records {P1}/splits/val.jsonl \
    --comparison-set /content/laya-vision/splits/val_comparison_2000_seed11.json --views text --out {P3}/baseline_untrained_text.json

## Ablations (20k × 2 epochs, ~25 min each)
One change per run. Results (combined / image on the monitor pool): tok8 0.764/0.752 · tok32 0.783/0.768 ·
tok64 0.789/0.785 · combined-only 0.809/0.688 · projector lr 3e-4 0.795/0.809. Text-only is 0.262 in every run.

In [ ]:
common = f"--base-ckpt {STEP5}/laya_finetuned --shards-root {P2}/shards --records-dir {P1}/splits --comparison-set /content/laya-vision/splits/val_comparison_2000_seed11.json --limit 20000 --epochs 2"
!python scripts/train_vision.py {common} --tokens 8  --out-dir {P3}/abl_tok8
!python scripts/train_vision.py {common} --tokens 32 --out-dir {P3}/abl_tok32
!python scripts/train_vision.py {common} --tokens 64 --out-dir {P3}/abl_tok64
!python scripts/train_vision.py {common} --modality-probs 1.0,0.0,0.0 --out-dir {P3}/abl_combined_only
!python scripts/train_vision.py {common} --lr-projector 0.0003 --out-dir {P3}/abl_lrproj3e-4

## Final run
Winners combined: 64 tokens, projector lr 3e-4, modality 40/30/30, 3 epochs → run02_final: monitor combined 0.872 /
image 0.859 / text 0.304. This is the Phase 4 checkpoint and the released model.

In [ ]:
!python scripts/train_vision.py --base-ckpt {STEP5}/laya_finetuned --shards-root {P2}/shards --records-dir {P1}/splits \
    --comparison-set /content/laya-vision/splits/val_comparison_2000_seed11.json --tokens 64 --lr-projector 0.0003 --epochs 3 --out-dir {P3}/run02_final

Gate 3 recorded in `DECISIONS.md`. Next: `phase4_colab_launcher.ipynb` (calibration and evaluation).